# The Sticky Price Tag: Tracing Global Commodity Shocks to the Grocery Store

This Colab notebook is the exploratory data analysis (EDA) deliverable for the project. It downloads upstream World Bank commodity prices and downstream BLS retail grocery prices, cleans them to a shared monthly timeline, normalizes each commodity-retail pair to a common baseline, and then builds three interactive Altair visualizations designed to explore price stickiness.


## Problem Context / Question

Price stickiness describes the pattern where consumer prices often rise quickly when upstream costs spike but fall much more slowly when those same upstream pressures ease. In grocery markets, that lag is especially visible: a consumer may hear that global wheat or coffee prices are cooling while bread and coffee still feel expensive at checkout.

This notebook investigates a simple question: **how many months separate a global commodity shock from the retail grocery response, and does the downward pass-through tend to be slower than the upward pass-through?**

To keep the analysis grounded in official, public data, the notebook pairs World Bank benchmark commodity series with BLS average retail price series and treats the resulting lag as an exploratory visual question rather than a claim of direct causality.


## Data Source Analysis

**World Bank Commodity Price Data (Pink Sheet):** The World Bank's monthly Pink Sheet provides long-run nominal U.S. dollar price benchmarks for raw commodities. In this notebook, those series represent the upstream portion of the food supply chain before retail packaging, distribution, and shelf-level pricing behavior.

**U.S. BLS Average Price Data:** The BLS Average Price program publishes monthly average prices paid by consumers for selected items. These series provide the downstream retail view that consumers actually experience.

**Important pairing note:** The current official World Bank monthly workbook exposes a `Beef **` benchmark in the monthly price sheet rather than a `Live cattle` column. To keep the notebook tied to the inspected official workbook structure, this EDA pairs **Beef / Ground Beef** instead of **Cattle / Beef**. The wheat/bread and coffee/roasted coffee pairings remain direct.


In [ ]:
# Colab usually includes pandas and requests already, but Altair and openpyxl are not
# guaranteed in every fresh runtime. Installing them here keeps the notebook portable.
%pip -q install altair openpyxl requests


In [ ]:
# Import the libraries used for data access, cleaning, transformation, and visualization.
# We keep the dependency list small on purpose so the notebook stays easy to rerun in Colab.
from datetime import date

import altair as alt
import numpy as np
import pandas as pd
import requests
from IPython.display import display

alt.data_transformers.disable_max_rows()

# These constants centralize the official source endpoints and the pair definitions so the
# rest of the notebook can stay clean and readable.
WORLD_BANK_URL = "https://thedocs.worldbank.org/en/doc/74e8be41ceb20fa0da750cda2f6b9e4e-0050012026/related/CMO-Historical-Data-Monthly.xlsx"
BLS_API_URL = "https://api.bls.gov/publicAPI/v2/timeseries/data/"
BASELINE_YEAR = 2010
CURRENT_YEAR = date.today().year

# Each record defines one commodity-retail pair. The commodity column names match the
# inspected World Bank workbook exactly, while the retail series ids come from the
# official BLS average-price series file.
PAIR_CONFIG = [
    {
        "Commodity_Pair": "Wheat/Bread",
        "commodity_col": "Wheat, US SRW",
        "commodity_label": "Wheat, US SRW",
        "retail_series_id": "APU0000702111",
        "retail_label": "Bread, white, pan",
    },
    {
        "Commodity_Pair": "Coffee/Roasted Coffee",
        "commodity_col": "Coffee, Arabica",
        "commodity_label": "Coffee, Arabica",
        "retail_series_id": "APU0000717311",
        "retail_label": "Coffee, 100%, ground roast, all sizes",
    },
    {
        "Commodity_Pair": "Beef/Ground Beef",
        "commodity_col": "Beef **",
        "commodity_label": "Beef benchmark",
        "retail_series_id": "APU0000703112",
        "retail_label": "Ground beef, 100% beef",
    },
]

pair_metadata = pd.DataFrame(PAIR_CONFIG)
PAIR_OPTIONS = pair_metadata["Commodity_Pair"].tolist()
DEFAULT_PAIR = PAIR_OPTIONS[0]

display(pair_metadata)


,Commodity_Pair,commodity_col,commodity_label,retail_series_id,retail_label
0,Wheat/Bread,"Wheat, US SRW","Wheat, US SRW",APU0000702111,"Bread, white, pan"
1,Coffee/Roasted Coffee,"Coffee, Arabica","Coffee, Arabica",APU0000717311,"Coffee, 100%, ground roast, all sizes"
2,Beef/Ground Beef,Beef **,Beef benchmark,APU0000703112,"Ground beef, 100% beef"


## Data Acquisition and Cleaning

The next cells do the heavy lifting for the EDA:

1. Download the World Bank monthly workbook and isolate the `Monthly Prices` sheet.
2. Pull BLS average retail price series from the official BLS API in 20-year chunks.
3. Convert both sources to a monthly `Date` field.
4. Standardize pair metadata, remove placeholder symbols, and create a merged analytic table.
5. Normalize each pair to a common baseline so upstream and downstream series can be compared visually.


In [ ]:
def load_world_bank_monthly_prices(url: str = WORLD_BANK_URL) -> pd.DataFrame:
    """Load the World Bank Monthly Prices sheet and keep only true monthly observations."""

    # The inspected workbook layout shows that row 5 contains commodity names and row 6
    # contains units. Using header=4 captures the commodity names as columns, and then we
    # filter away the units row by keeping only values that look like YYYYMmm dates.
    raw = pd.read_excel(url, sheet_name="Monthly Prices", header=4, engine="openpyxl")

    # The first unnamed column is the workbook's month label column. Renaming it to Date
    # makes the downstream pipeline much easier to read.
    raw = raw.rename(columns={raw.columns[0]: "Date"})

    # The workbook uses both the unicode ellipsis character and three dots as placeholders
    # for unavailable values. Replacing them with NaN ensures pandas can treat the columns
    # as numeric after conversion.
    cleaned = raw.replace({"…": np.nan, "...": np.nan})

    # Only rows whose Date looks like 1960M01, 1960M02, etc. are actual monthly records.
    cleaned = cleaned.loc[cleaned["Date"].astype(str).str.match(r"^\d{4}M\d{2}$")].copy()

    # Convert strings like 1960M01 into real month-start timestamps so Altair and pandas
    # can work with them as a proper time axis.
    cleaned["Date"] = pd.to_datetime(cleaned["Date"].str.replace("M", "-", regex=False) + "-01")

    # We only need the inspected commodity columns used in the chosen pairs. Keeping the
    # subset small makes later reshaping simpler and reduces accidental type issues.
    keep_cols = ["Date"] + pair_metadata["commodity_col"].tolist()
    cleaned = cleaned[keep_cols].copy()

    # Force the selected commodity columns to numeric after placeholder cleanup. Any value
    # that still fails conversion becomes NaN rather than breaking the notebook.
    for col in pair_metadata["commodity_col"]:
        cleaned[col] = pd.to_numeric(cleaned[col], errors="coerce")

    return cleaned.sort_values("Date").reset_index(drop=True)


def fetch_bls_average_prices(series_ids, start_year: int = 1980, end_year: int = CURRENT_YEAR) -> pd.DataFrame:
    """Pull monthly BLS average-price data in API-safe chunks."""

    rows = []

    # The public BLS API is limited to relatively short year spans per request. Looping in
    # 20-year windows keeps the notebook compatible with the public endpoint without asking
    # for an API key.
    for chunk_start in range(start_year, end_year + 1, 20):
        chunk_end = min(chunk_start + 19, end_year)
        payload = {
            "seriesid": series_ids,
            "startyear": str(chunk_start),
            "endyear": str(chunk_end),
        }

        response = requests.post(BLS_API_URL, json=payload, timeout=60)
        response.raise_for_status()
        body = response.json()

        if body.get("status") != "REQUEST_SUCCEEDED":
            raise RuntimeError(f"BLS API request failed: {body}")

        # Each series comes back as a nested list of monthly observations. We keep only
        # true months (M01-M12) and ignore annual summary entries like M13.
        for series in body["Results"]["series"]:
            series_id = series["seriesID"]
            for obs in series["data"]:
                period = obs["period"]
                if not period.startswith("M") or period == "M13":
                    continue
                rows.append(
                    {
                        "retail_series_id": series_id,
                        "year": int(obs["year"]),
                        "month": int(period[1:]),
                        "Date": pd.Timestamp(year=int(obs["year"]), month=int(period[1:]), day=1),
                        "Retail_Price": float(obs["value"]),
                        "footnotes": ", ".join(
                            ft.get("text", "")
                            for ft in obs.get("footnotes", [])
                            if ft and ft.get("text")
                        ),
                    }
                )

    bls = pd.DataFrame(rows).sort_values(["retail_series_id", "Date"]).reset_index(drop=True)
    return bls


In [ ]:
def add_normalized_index(df: pd.DataFrame, value_col: str, baseline_year: int = BASELINE_YEAR) -> pd.DataFrame:
    """Normalize each pair to a baseline year so unlike units can be compared on one scale."""

    normalized_parts = []

    # We normalize within each pair rather than across the entire dataset because wheat,
    # coffee, and beef have different natural price levels and units.
    for pair_name, group in df.groupby("Commodity_Pair", sort=False):
        group = group.copy()

        # A baseline-year mean gives a stable reference level. If a pair has no baseline-year
        # observations, we fall back to the first non-null observation so the notebook still runs.
        baseline = group.loc[group["Date"].dt.year == baseline_year, value_col].mean()
        if pd.isna(baseline):
            baseline = group[value_col].dropna().iloc[0]

        group[f"{value_col}_Index"] = group[value_col] / baseline * 100
        normalized_parts.append(group)

    return pd.concat(normalized_parts, ignore_index=True)


def compute_lag_scan(df: pd.DataFrame, max_lag: int = 12) -> pd.DataFrame:
    """Create a heuristic lead-lag table using month-over-month normalized changes."""

    lag_rows = []

    # This is exploratory only. A high correlation at a given lag is a useful clue for EDA,
    # but it should not be interpreted as proof of causality.
    for pair_name, group in df.groupby("Commodity_Pair", sort=False):
        group = group.sort_values("Date").copy()
        commodity_change = group["Commodity_Price_Index"].pct_change(fill_method=None)
        retail_change = group["Retail_Price_Index"].pct_change(fill_method=None)

        for lag in range(max_lag + 1):
            lag_rows.append(
                {
                    "Commodity_Pair": pair_name,
                    "Lag_Months": lag,
                    "Correlation": commodity_change.shift(lag).corr(retail_change),
                }
            )

    return pd.DataFrame(lag_rows)

def fetch_bls_average_prices(series_ids, start_year: int = 1980, end_year: int = CURRENT_YEAR) -> pd.DataFrame:
    """Pull monthly BLS average-price data in API-safe chunks."""

    rows = []

    # The public BLS API is limited to relatively short year spans per request. Looping in
    # 20-year windows keeps the notebook compatible with the public endpoint without asking
    # for an API key.
    for chunk_start in range(start_year, end_year + 1, 20):
        chunk_end = min(chunk_start + 19, end_year)
        payload = {
            "seriesid": series_ids,
            "startyear": str(chunk_start),
            "endyear": str(chunk_end),
        }

        response = requests.post(BLS_API_URL, json=payload, timeout=60)
        response.raise_for_status()
        body = response.json()

        if body.get("status") != "REQUEST_SUCCEEDED":
            raise RuntimeError(f"BLS API request failed: {body}")

        # Each series comes back as a nested list of monthly observations. We keep only
        # true months (M01-M12) and ignore annual summary entries like M13.
        for series in body["Results"]["series"]:
            series_id = series["seriesID"]
            for obs in series["data"]:
                period = obs["period"]
                if not period.startswith("M") or period == "M13":
                    continue
                rows.append(
                    {
                        "retail_series_id": series_id,
                        "year": int(obs["year"]),
                        "month": int(period[1:]),
                        "Date": pd.Timestamp(year=int(obs["year"]), month=int(period[1:]), day=1),
                        "Retail_Price": pd.to_numeric(obs["value"], errors="coerce"), # Changed to handle non-numeric values
                        "footnotes": ", ".join(
                            ft.get("text", "")
                            for ft in obs.get("footnotes", [])
                            if ft and ft.get("text")
                        ),
                    }
                )

    bls = pd.DataFrame(rows).sort_values(["retail_series_id", "Date"]).reset_index(drop=True)
    return bls


world_bank_prices = load_world_bank_monthly_prices()
bls_prices = fetch_bls_average_prices(pair_metadata["retail_series_id"].tolist())

# Reshape the selected World Bank commodity columns from wide format to long format so each
# row represents one monthly commodity observation for one pair.
commodity_long = (
    world_bank_prices
    .melt(id_vars="Date", var_name="commodity_col", value_name="Commodity_Price")
    .dropna(subset=["Commodity_Price"])
    .merge(pair_metadata[["Commodity_Pair", "commodity_col", "commodity_label", "retail_series_id", "retail_label"]], on="commodity_col", how="left")
)

# Attach the pair metadata to the retail table so the merge can happen on Date and pair.
retail_long = (
    bls_prices
    .merge(pair_metadata[["Commodity_Pair", "retail_series_id", "retail_label"]], on="retail_series_id", how="inner")
    .sort_values(["Commodity_Pair", "Date"])
    .reset_index(drop=True)
)

# Merge upstream commodity prices with downstream retail prices on the same monthly timeline.
# We use an inner join because the paired charts should only display months where both series exist.
merged = (
    commodity_long
    .merge(retail_long[["Date", "Commodity_Pair", "Retail_Price", "retail_label"]], on=["Date", "Commodity_Pair"], how="inner")
    .sort_values(["Commodity_Pair", "Date"])
    .reset_index(drop=True)
)

# Normalize both the commodity and retail price levels so they share a baseline of 100.
# This makes the line charts visually comparable even though the raw series use different units.
merged = add_normalized_index(merged, "Commodity_Price", baseline_year=BASELINE_YEAR)
merged = add_normalized_index(merged, "Retail_Price", baseline_year=BASELINE_YEAR)

# The gap between the two normalized indexes is a convenient exploratory measure for the
# margin-style chart because it compares relative movement rather than incompatible units.
merged["Normalized_Gap"] = merged["Retail_Price_Index"] - merged["Commodity_Price_Index"]

# Month-over-month percent changes let us build the volatility and stickiness heatmap.
merged["Commodity_MoM_Pct"] = merged.groupby("Commodity_Pair")["Commodity_Price"].pct_change(fill_method=None) * 100
merged["Retail_MoM_Pct"] = merged.groupby("Commodity_Pair")["Retail_Price"].pct_change(fill_method=None) * 100

# Create a small coverage table so the notebook immediately tells us how much history is
# available for each pair after the merge.
coverage_summary = (
    merged.groupby("Commodity_Pair")
    .agg(
        Start_Date=("Date", "min"),
        End_Date=("Date", "max"),
        Monthly_Observations=("Date", "count"),
    )
    .reset_index()
)

# Build a heuristic lag table to give the EDA a quantitative starting point. Again, this is
# not the final answer; it simply suggests which lags are worth inspecting visually.
lag_scan = compute_lag_scan(merged, max_lag=12)
lag_summary = (
    lag_scan
    .dropna(subset=["Correlation"])
    .sort_values(["Commodity_Pair", "Correlation"], ascending=[True, False])
    .groupby("Commodity_Pair", as_index=False)
    .head(1)
    .rename(columns={"Correlation": "Best_Correlation"})
    .sort_values("Commodity_Pair")
    .reset_index(drop=True)
)

display(coverage_summary)
display(lag_summary)
display(merged.head())

,Commodity_Pair,Start_Date,End_Date,Monthly_Observations
0,Beef/Ground Beef,1984-01-01,2026-04-01,268
1,Coffee/Roasted Coffee,1980-01-01,2026-04-01,291
2,Wheat/Bread,1980-01-01,2026-04-01,309


,Commodity_Pair,Lag_Months,Best_Correlation
0,Beef/Ground Beef,0,0.719226
1,Coffee/Roasted Coffee,2,0.322219
2,Wheat/Bread,5,0.102034


,Date,commodity_col,Commodity_Price,Commodity_Pair,commodity_label,retail_series_id,retail_label_x,Retail_Price,retail_label_y,Commodity_Price_Index,Retail_Price_Index,Normalized_Gap,Commodity_MoM_Pct,Retail_MoM_Pct
0,1984-01-01,Beef **,2.2619,Beef/Ground Beef,Beef benchmark,APU0000703112,"Ground beef, 100% beef",1.290,"Ground beef, 100% beef",100.000000,100.000000,0.000000,NaN,NaN
1,1984-02-01,Beef **,2.3390,Beef/Ground Beef,Beef benchmark,APU0000703112,"Ground beef, 100% beef",1.340,"Ground beef, 100% beef",103.408639,103.875969,0.467330,3.408639,3.875969
2,1984-03-01,Beef **,2.3810,Beef/Ground Beef,Beef benchmark,APU0000703112,"Ground beef, 100% beef",1.308,"Ground beef, 100% beef",105.265485,101.395349,-3.870136,1.795639,-2.388060
3,1984-04-01,Beef **,2.4692,Beef/Ground Beef,Beef benchmark,APU0000703112,"Ground beef, 100% beef",1.331,"Ground beef, 100% beef",109.164861,103.178295,-5.986567,3.704326,1.758410
4,1984-05-01,Beef **,2.3951,Beef/Ground Beef,Beef benchmark,APU0000703112,"Ground beef, 100% beef",1.301,"Ground beef, 100% beef",105.888855,100.852713,-5.036141,-3.000972,-2.253944


## Exploratory Snapshot

The merged table above is the working backbone for the rest of the notebook. The lag summary is intentionally heuristic: it uses month-over-month normalized changes to suggest which lags deserve closer visual inspection. The interactive charts below remain the primary way to investigate the time-delay story.


## Visualization Breakdown 1: The Delay Detector

### Target Audience
Everyday consumers who want to connect news about global commodity prices to their own grocery receipts, plus students learning how lagged pass-through appears in time-series data.

### Task
Use the contextual brush at the bottom of the chart to zoom into a specific shock window and visually compare when the upstream commodity turns versus when the retail series follows.

### Key Insights
Users should look for months where the commodity index changes direction before the retail index, and especially for periods where retail prices stay elevated after the commodity cools.

### Unexpected Learnings
This section should be updated after notebook execution. In practice, surprising findings may include retail persistence during falling commodity prices or weak retail reactions to sharp but short-lived commodity shocks.


In [ ]:
# Convert the merged wide table into a long plotting table so Altair can draw the commodity
# and retail index as two lines on the same axes.
# Note: We use retail_label_x which was created during the merge in the previous cell.
delay_detector_source = pd.concat(
    [
        merged[["Date", "Commodity_Pair", "commodity_label", "Commodity_Price_Index"]]
        .rename(columns={"commodity_label": "Series_Label", "Commodity_Price_Index": "Index_Value"})
        .assign(Series="Commodity"),
        merged[["Date", "Commodity_Pair", "retail_label_x", "Retail_Price_Index"]]
        .rename(columns={"retail_label_x": "Series_Label", "Retail_Price_Index": "Index_Value"})
        .assign(Series="Retail"),
    ],
    ignore_index=True,
)


def build_delay_detector(df: pd.DataFrame) -> alt.Chart:
    selected_pair = alt.param(
        name="selected_pair_delay",
        value=DEFAULT_PAIR,
        bind=alt.binding_select(options=PAIR_OPTIONS, name="Commodity Pair: "),
    )

    # The interval brush on the lower chart creates the zoom domain for the main chart.
    brush = alt.selection_interval(encodings=["x"], name="delay_window")
    pair_filter = f"datum.Commodity_Pair == {selected_pair.name}"

    base = (
        alt.Chart(df)
        .transform_filter(pair_filter)
        .encode(
            x=alt.X("Date:T", title="Month"),
            color=alt.Color(
                "Series:N",
                title=None,
                scale=alt.Scale(domain=["Commodity", "Retail"], range=["#b85d3d", "#305e4b"]),
            ),
            tooltip=[
                alt.Tooltip("Commodity_Pair:N", title="Pair"),
                alt.Tooltip("Series:N", title="Series"),
                alt.Tooltip("Series_Label:N", title="Label"),
                alt.Tooltip("Date:T", title="Month"),
                alt.Tooltip("Index_Value:Q", title="Normalized Index", format=".2f"),
            ],
        )
    )

    focus = (
        base.mark_line(strokeWidth=2.8)
        .encode(
            x=alt.X("Date:T", scale=alt.Scale(domain=brush)),
            y=alt.Y("Index_Value:Q", title=f"Normalized price index ({BASELINE_YEAR}=100)"),
        )
        .properties(width=760, height=320, title="The Delay Detector")
    )

    context = (
        base.mark_line(opacity=0.75)
        .encode(y=alt.Y("Index_Value:Q", title=None))
        .add_params(selected_pair, brush)
        .properties(width=760, height=90, title="Brush this lower timeline to zoom the main chart")
    )

    return alt.vconcat(focus, context)


build_delay_detector(delay_detector_source)

alt.VConcatChart(...)

## Visualization Breakdown 2: The Margin Explorer

### Target Audience
Economics students and analysts who want to study how upstream and downstream prices move together, and when they visibly diverge.

### Task
Brush a cluster of points in the scatter plot to isolate a regime, such as high retail prices paired with comparatively lower commodity prices, and then use the linked area chart to see when those months occurred.

### Key Insights
Users should look for clusters that suggest abnormal divergence, then check whether those months line up with recession windows, supply-chain shocks, or slower retail pass-through.

### Unexpected Learnings
This section should be updated after execution. A likely surprise is that the widest price gaps may occur after the initial commodity spike, once the upstream market begins to cool but retail prices remain sticky.


In [ ]:
# The scatter uses raw price levels on separate axes so the user can inspect the relationship
# between upstream and downstream prices. The linked area chart uses the normalized gap so the
# y-axis captures relative divergence instead of incomparable units.
margin_source = merged.copy()


def build_margin_explorer(df: pd.DataFrame) -> alt.Chart:
    selected_pair = alt.param(
        name="selected_pair_margin",
        value=DEFAULT_PAIR,
        bind=alt.binding_select(options=PAIR_OPTIONS, name="Commodity Pair: "),
    )
    pair_filter = f"datum.Commodity_Pair == {selected_pair.name}"
    scatter_brush = alt.selection_interval(name="margin_cluster")

    scatter = (
        alt.Chart(df)
        .transform_filter(pair_filter)
        .mark_circle(size=90, opacity=0.8)
        .encode(
            x=alt.X("Commodity_Price:Q", title="Commodity benchmark price"),
            y=alt.Y("Retail_Price:Q", title="Retail average price"),
            color=alt.condition(scatter_brush, alt.value("#b85d3d"), alt.value("#305e4b")),
            tooltip=[
                alt.Tooltip("Commodity_Pair:N", title="Pair"),
                alt.Tooltip("Date:T", title="Month"),
                alt.Tooltip("Commodity_Price:Q", title="Commodity Price", format=".2f"),
                alt.Tooltip("Retail_Price:Q", title="Retail Price", format=".2f"),
                alt.Tooltip("Normalized_Gap:Q", title="Normalized Gap", format=".2f"),
            ],
        )
        .add_params(selected_pair, scatter_brush)
        .properties(width=360, height=320, title="Raw Price Relationship")
    )

    gap_chart = (
        alt.Chart(df)
        .transform_filter(pair_filter)
        .transform_filter(scatter_brush)
        .mark_area(color="#b85d3d", opacity=0.55)
        .encode(
            x=alt.X("Date:T", title="Month"),
            y=alt.Y("Normalized_Gap:Q", title="Retail minus commodity (normalized index points)"),
            tooltip=[
                alt.Tooltip("Date:T", title="Month"),
                alt.Tooltip("Normalized_Gap:Q", title="Normalized Gap", format=".2f"),
            ],
        )
        .properties(width=360, height=320, title="Price Gap Over Time for the Selected Cluster")
    )

    return scatter | gap_chart


build_margin_explorer(margin_source)


alt.HConcatChart(...)

## Visualization Breakdown 3: Volatility & Stickiness Viewer

### Target Audience
Users who want to compare sector behavior without overplotting several unrelated commodity pairs in one chart.

### Task
Use the dropdown to switch between commodity-retail pairs and inspect month-over-month changes. Look for cases where commodity volatility fades quickly but retail cooling remains weak or delayed.

### Key Insights
Users should compare how quickly different sectors transmit shocks, how persistent negative retail corrections are, and whether volatility in raw inputs translates into equally volatile shelf prices.

### Unexpected Learnings
This section should be updated after execution. One likely finding is that the most volatile commodity pair is not automatically the stickiest retail pair.


In [ ]:
# Build a compact long table with one row per pair, month, and series type so the heatmap
# can compare commodity and retail month-over-month changes side by side.
volatility_source = pd.concat(
    [
        merged[["Date", "Commodity_Pair", "Commodity_MoM_Pct"]]
        .rename(columns={"Commodity_MoM_Pct": "MoM_Pct"})
        .assign(Series="Commodity MoM %"),
        merged[["Date", "Commodity_Pair", "Retail_MoM_Pct"]]
        .rename(columns={"Retail_MoM_Pct": "MoM_Pct"})
        .assign(Series="Retail MoM %"),
    ],
    ignore_index=True,
).dropna(subset=["MoM_Pct"])


def build_volatility_viewer(df: pd.DataFrame) -> alt.Chart:
    selected_pair = alt.param(
        name="selected_pair_volatility",
        value=DEFAULT_PAIR,
        bind=alt.binding_select(options=PAIR_OPTIONS, name="Commodity Pair: "),
    )
    pair_filter = f"datum.Commodity_Pair == {selected_pair.name}"

    heatmap = (
        alt.Chart(df)
        .transform_filter(pair_filter)
        .mark_rect(stroke="white")
        .encode(
            x=alt.X("yearmonth(Date):T", title="Month"),
            y=alt.Y("Series:N", title=None, sort=["Commodity MoM %", "Retail MoM %"]),
            color=alt.Color(
                "MoM_Pct:Q",
                title="Month-over-month % change",
                scale=alt.Scale(scheme="redblue", domainMid=0),
            ),
            tooltip=[
                alt.Tooltip("Commodity_Pair:N", title="Pair"),
                alt.Tooltip("Series:N", title="Series"),
                alt.Tooltip("Date:T", title="Month"),
                alt.Tooltip("MoM_Pct:Q", title="MoM % Change", format=".2f"),
            ],
        )
        .add_params(selected_pair)
        .properties(width=780, height=120, title="Volatility & Stickiness Viewer")
    )

    return heatmap


build_volatility_viewer(volatility_source)


alt.Chart(...)

## Usability Testing Protocol

Before the notebook is finalized, at least **two different test subjects** should interact with the charts.

**Recommended participant mix**
- One everyday consumer who is not already familiar with commodity pricing.
- One economics or data-literate participant who can speak to analytical clarity.

**Tasks for the test subjects**
1. Use the Delay Detector to estimate the lag for one major macro event.
2. Use the Margin Explorer to isolate a period of extreme divergence and identify when it happened.
3. Use the Volatility & Stickiness Viewer to compare which pair looks most sticky after a shock cools.

**What to document after testing**
- Which interactions were confusing.
- Whether the normalization language was clear.
- Whether the scatter-to-area cross-filter was intuitive.
- Whether chart titles, legends, and axes need revision.

Once the live test is completed, replace this protocol-only section with a short findings log and a record of the design changes that were made.


## Concluding Comments

This notebook is designed to test the hypothesis that grocery prices often respond to commodity shocks with a noticeable lag and that retail price declines may trail commodity declines more slowly than retail price increases trail commodity spikes.

The notebook does **not** assume that every retail delay is caused by retailer behavior alone. The visual gaps may also reflect contracts, freight, labor, packaging, inventory timing, and imperfect pair definitions. For that reason, the charts are framed as exploratory tools for detecting patterns and generating more careful interpretation.

The next step after running the notebook is to record the actual findings for each pair, document the usability test revisions, and export final chart screenshots into the presentation/report deliverables.
